In [2]:
# =====================================================================
# Sprint 54 · Task 1 — Judge fault-sensitivity test (EN→CMN)  — v2
# Free open-weight judges via Unsloth. Colab T4 (free tier) is enough.
# Runtime → Change runtime type → GPU, then run top-to-bottom.
#
# v2 changes: reproducible perturbations · fixed sentence set shared by all
# judges · 3 samples per item (median) · control-adjusted drop · bootstrap CIs
# · fluency gate on meaning-altering edits · unparsed-rate guard · raw outputs
# kept · transformers fallback if Unsloth can't load a model.
#
# Outputs: results_all_judges.csv, summary_all_judges.csv, judge_comparison.csv, FINDING.md
# =====================================================================
!pip -q install unsloth sacrebleu jieba pandas numpy tqdm tabulate

import os, re, json, random, hashlib, subprocess, statistics, gc, math
import numpy as np, torch, pandas as pd
from tqdm.auto import tqdm
import jieba, jieba.posseg as pseg
jieba.setLogLevel(60)

# ----------------------------- CONFIG --------------------------------
MODELS = [
    {"name": "qwen3-8b",   "id": "unsloth/Qwen3-8B-unsloth-bnb-4bit",
     "template_kwargs": {"enable_thinking": False}},
    {"name": "glm4-9b",    "id": "unsloth/GLM-4-9B-0414", "load_in_4bit": True},
    {"name": "qwen2.5-7b", "id": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"},
]
FLUENCY_GATE_MODEL = 0        # index into MODELS; runs the grammar gate once for everyone

N_CANDIDATES   = 150          # sentences sampled from the dataset
HIGH_SCORE_MIN = 8            # "scored highly" threshold (median of samples)
MAX_KEPT       = 60           # cap on sentences carried into perturbation
MIN_KEPT       = 20           # below this the shared set is widened (see select_kept)
N_SAMPLES      = 3            # judge samples per item; median is used
TEMPERATURE    = 0.3
SEED           = 54
BATCH_SIZE     = 6            # prompts per batch (× N_SAMPLES sequences)
MAX_NEW_TOKENS = 96
MAX_SEQ_LEN    = 2048
CACHE_FILE     = "judge_cache_v2.json"
BOOT_N         = 2000

EXPECTED_MIN_DROP = {"untranslated": 7, "negation": 5, "entity": 4, "clause": 3, "truncation": 3}
CONTROL_MAX_DROP  = 1
RELIABLE_PCT      = 80        # point estimate must reach this ...
RELIABLE_CI_LO    = 65        # ... and the 95% CI lower bound must reach this
MAX_UNPARSED_PCT  = 20        # judge flagged UNUSABLE above this
GATED_FAULTS      = {"negation", "entity", "clause"}   # edits that must still read as fluent Chinese

# >>> Paste your production judge prompt here. Keep {src} and {cand}. <<<
JUDGE_PROMPT = """You are an expert evaluator of English-to-Mandarin Chinese translation.

Rate the candidate translation on a 0-10 integer scale for how well it conveys the source
(accuracy of meaning, completeness, and fluency). 10 = a professional-quality translation;
0 = conveys none of the source meaning.

Source (English):
{src}

Candidate (Chinese):
{cand}

Respond with JSON only, no prose: {{"score": <integer 0-10>, "reason": "<one sentence>"}}"""

FLUENCY_PROMPT = """You will see one sentence in Chinese. Ignore its meaning and do not compare it to anything.
Judge only whether it is grammatical, natural Chinese that a native speaker could plausibly have written.

Sentence:
{cand}

Respond with JSON only: {{"fluent": true}} or {{"fluent": false}}"""

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU (Runtime → Change runtime type)."

# --------------------------- DATASET ---------------------------------
def load_wmt(testset="wmt20", langpair="en-zh"):
    def echo(what):
        r = subprocess.run(["sacrebleu", "-t", testset, "-l", langpair, "--echo", what],
                           capture_output=True, text=True, check=True)
        return r.stdout.rstrip("\n").split("\n")
    src, ref = echo("src"), echo("ref")
    assert len(src) == len(ref) and len(src) > 100
    return list(zip(src, ref)), f"WMT20 newstest en-zh (statmt.org via sacrebleu), {len(src)} segments"

def load_opus100():
    from datasets import load_dataset
    ds = load_dataset("Helsinki-NLP/opus-100", "en-zh", split="test")
    return [(r["translation"]["en"], r["translation"]["zh"]) for r in ds], "OPUS-100 en-zh test (HuggingFace)"

try:
    all_pairs, DATASET_NAME = load_wmt()
except Exception as e:
    print("WMT20 load failed, falling back to OPUS-100:", repr(e)[:200])
    all_pairs, DATASET_NAME = load_opus100()

def usable(en, zh):
    return (8 <= len(en.split()) <= 45 and len(zh) >= 10
            and re.search(r"[\u4e00-\u9fff]", zh) is not None
            and not re.search(r"[A-Za-z]{3,}", zh))
pairs = [p for p in all_pairs if usable(*p)]
random.Random(SEED).shuffle(pairs)
pairs = pairs[:N_CANDIDATES]
print(f"Dataset: {DATASET_NAME}\nCandidates after filtering: {len(pairs)}")

# -------------------------- PERTURBATIONS ----------------------------
def stable_rng(*parts):                       # reproducible across processes (fix for salted hash())
    h = hashlib.md5("||".join(map(str, parts)).encode()).hexdigest()
    return random.Random(int(h, 16) ^ SEED)

def tokens_with_pos(zh):
    toks, i = [], 0
    for t in pseg.lcut(zh):
        toks.append((t.word, t.flag, i)); i += len(t.word)
    return toks

NEG_TOKENS = {"不", "没", "没有", "无法", "并非", "未", "并不", "不会", "不能", "不是"}
def perturb_negation(zh, en):
    toks = tokens_with_pos(zh)
    for w, f, i in toks:
        if w in NEG_TOKENS:
            return zh[:i] + zh[i+len(w):], f"removed negation '{w}' at char {i}"
    for w, f, i in toks:
        if f.startswith("v") and f != "vn":
            neg = "没" if w.startswith("有") else "不"
            return zh[:i] + neg + zh[i:], f"inserted '{neg}' before verb '{w}' at char {i}"
    return None

CN_NUM = "二三四五六七八九"
CN_SWAP = {c: CN_NUM[(k + 3) % len(CN_NUM)] for k, c in enumerate(CN_NUM)}
ENTITY_POOL = {"nr": ["李明", "王芳", "张伟", "刘洋"],
               "ns": ["北京", "伦敦", "东京", "柏林", "巴黎", "开罗"],
               "nt": ["联合国", "欧盟", "世界银行", "微软", "国际奥委会"]}
def perturb_entity(zh, en):
    toks = tokens_with_pos(zh)
    for w, f, i in toks:
        if f == "m":
            m = re.search(r"\d+", w)
            if m:
                old = int(m.group())
                new = old + stable_rng(zh, w).choice([2, 3, 5]) * (1 if old < 50 else 10)   # plausible-looking change
                new_w = w[:m.start()] + str(new) + w[m.end():]
                return zh[:i] + new_w + zh[i+len(w):], f"number '{w}' -> '{new_w}'"
            for ch in w:
                if ch in CN_SWAP:
                    new_w = w.replace(ch, CN_SWAP[ch], 1)
                    return zh[:i] + new_w + zh[i+len(w):], f"numeral '{w}' -> '{new_w}'"
    for w, f, i in toks:
        if f in ENTITY_POOL and len(w) >= 2:
            new_w = stable_rng(zh, w).choice([c for c in ENTITY_POOL[f] if c != w])
            return zh[:i] + new_w + zh[i+len(w):], f"entity[{f}] '{w}' -> '{new_w}'"
    return None

def perturb_clause(zh, en):
    clauses, buf = [], ""
    for ch in zh:
        buf += ch
        if ch in "，；：":
            clauses.append(buf); buf = ""
    if buf: clauses.append(buf)
    if len(clauses) < 2: return None
    droppable = [k for k, c in enumerate(clauses[:-1]) if len(c.strip("，；：")) >= 4]
    if not droppable: return None
    k = droppable[len(droppable) // 2]
    return "".join(c for j, c in enumerate(clauses) if j != k), f"dropped clause #{k+1}/{len(clauses)}: '{clauses[k]}'"

def perturb_truncation(zh, en):
    if len(zh) < 14: return None
    cut = int(len(zh) * 0.6)
    return zh[:cut], f"kept first {cut}/{len(zh)} chars (dropped {len(zh)-cut})"

def perturb_untranslated(zh, en):
    return en, "returned the English source verbatim as the output"

def perturb_control(zh, en):
    if "，" in zh:        return zh.replace("，", ",", 1), "first full-width comma '，' -> ASCII ','"
    if zh.endswith("。"): return zh[:-1] + ".", "final '。' -> ASCII '.'"
    return zh + "。", "appended final '。'"

FAULTS = {"untranslated": perturb_untranslated, "negation": perturb_negation, "entity": perturb_entity,
          "clause": perturb_clause, "truncation": perturb_truncation, "control": perturb_control}

perturb_table = []   # fixed for all judges
for k, (en, zh) in enumerate(pairs):
    for fault, fn in FAULTS.items():
        res = fn(zh, en)
        perturb_table.append(dict(sent_id=k, fault=fault, perturbed=res[0] if res else None,
                                  modification=res[1] if res else "SKIPPED: fault not applicable to this sentence"))

# ----------------------------- MODELS --------------------------------
_cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def load_model(cfg):
    try:
        from unsloth import FastLanguageModel, FastModel
        loader = FastModel if cfg.get("loader") == "FastModel" else FastLanguageModel
        model, tok = loader.from_pretrained(model_name=cfg["id"], max_seq_length=MAX_SEQ_LEN,
                                            load_in_4bit=cfg.get("load_in_4bit", True), dtype=None)
        loader.for_inference(model)
    except Exception as e:                                     # fix 3: plain transformers 4-bit fallback
        print(f"[{cfg['name']}] Unsloth loader failed ({repr(e)[:150]}) → transformers 4-bit fallback")
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        tok = AutoTokenizer.from_pretrained(cfg["id"])
        model = AutoModelForCausalLM.from_pretrained(cfg["id"], quantization_config=bnb, device_map="auto")
        model.eval()
    if getattr(tok, "pad_token", None) is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return model, tok

def unload(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def generate(model, tok, prompts, cfg, n=1, temperature=0.0):
    """Returns list (per prompt) of n decoded completions."""
    texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                     add_generation_prompt=True, **cfg.get("template_kwargs", {})) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id, num_return_sequences=n)
    kw.update(dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    dec = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [dec[i*n:(i+1)*n] for i in range(len(prompts))]

def parse_score(txt):
    txt = re.sub(r"<think>.*?</think>", "", txt, flags=re.S)
    m = re.search(r"\{.*?\}", txt, re.S)
    try:
        return max(0, min(10, int(round(float(json.loads(m.group(0))["score"])))))
    except Exception:
        m = re.search(r'"?score"?\s*[:=]\s*(\d+)', txt)
        return max(0, min(10, int(m.group(1)))) if m else None

def parse_fluent(txt):
    m = re.search(r'"?fluent"?\s*:\s*(true|false)', txt, re.I)
    return None if not m else m.group(1).lower() == "true"

def run_prompts(model, tok, cfg, jobs, prompt_tmpl, parser, n, temperature, desc, tag):
    """jobs: list of (jid, dict_of_prompt_fields) -> {jid: {"vals": [...parsed...], "raw": [...]}}"""
    out, todo = {}, []
    for jid, fields in jobs:
        key = hashlib.sha256(f"{tag}|{cfg['id']}|{prompt_tmpl}|{n}|{temperature}|{json.dumps(fields, ensure_ascii=False)}".encode()).hexdigest()
        if key in _cache: out[jid] = _cache[key]
        else: todo.append((jid, key, prompt_tmpl.format(**fields)))
    todo.sort(key=lambda t: len(t[2]))
    for b in tqdm(range(0, len(todo), BATCH_SIZE), desc=desc, leave=False):
        batch = todo[b:b+BATCH_SIZE]
        for (jid, key, _), raws in zip(batch, generate(model, tok, [t[2] for t in batch], cfg, n, temperature)):
            rec = {"vals": [parser(r) for r in raws], "raw": raws}
            _cache[key] = rec; out[jid] = rec
    json.dump(_cache, open(CACHE_FILE, "w"), ensure_ascii=False)
    return out

def median_score(rec):
    vals = [v for v in rec["vals"] if v is not None]
    return statistics.median(vals) if len(vals) >= math.ceil(len(rec["vals"]) / 2) else None

# ---------------- PHASE 1: originals (all judges) + fluency gate -------------
orig = {}        # judge name -> {sent_id: median score}
orig_raw = {}
for mi, cfg in enumerate(MODELS):
    try:
        model, tok = load_model(cfg)
        res = run_prompts(model, tok, cfg, [(k, {"src": en, "cand": zh}) for k, (en, zh) in enumerate(pairs)],
                          JUDGE_PROMPT, parse_score, N_SAMPLES, TEMPERATURE, f"[{cfg['name']}] originals", "judge")
        orig[cfg["name"]] = {k: median_score(r) for k, r in res.items()}
        orig_raw[cfg["name"]] = res
        if mi == FLUENCY_GATE_MODEL:                                   # fix 6: fluency gate, run once
            gate_jobs = [(i, {"cand": row["perturbed"]}) for i, row in enumerate(perturb_table)
                         if row["perturbed"] is not None and row["fault"] in GATED_FAULTS]
            gres = run_prompts(model, tok, cfg, gate_jobs, FLUENCY_PROMPT, parse_fluent, 1, 0.0,
                               f"[{cfg['name']}] fluency gate", "gate")
            for i, r in gres.items():
                perturb_table[i]["fluent"] = r["vals"][0]
        unload(model)
        s = [v for v in orig[cfg["name"]].values() if v is not None]
        print(f"[{cfg['name']}] originals: mean {statistics.mean(s):.2f}, std {statistics.pstdev(s):.2f}, "
              f"min {min(s)}, max {max(s)}, >= {HIGH_SCORE_MIN}: {sum(v >= HIGH_SCORE_MIN for v in s)}, unparsed {len(pairs)-len(s)}")
    except Exception as e:
        print(f"[{cfg['name']}] FAILED in phase 1: {repr(e)[:300]}"); gc.collect(); torch.cuda.empty_cache()
assert orig, "No judge completed phase 1."
JUDGES = [c for c in MODELS if c["name"] in orig]
for row in perturb_table: row.setdefault("fluent", None)   # None = not gated (untranslated/truncation/control) or unparsed

# fix 2: one shared sentence set
def select_kept():
    high = {c["name"]: {k for k, v in orig[c["name"]].items() if v is not None and v >= HIGH_SCORE_MIN} for c in JUDGES}
    inter = set.intersection(*high.values())
    note = f"sentences scored >= {HIGH_SCORE_MIN} by EVERY judge"
    if len(inter) < MIN_KEPT:
        inter = {k for k in range(len(pairs))
                 if all(orig[c["name"]].get(k) is not None for c in JUDGES)
                 and statistics.mean(orig[c["name"]][k] for c in JUDGES) >= HIGH_SCORE_MIN}
        note = f"intersection too small; widened to sentences with MEAN judge score >= {HIGH_SCORE_MIN}"
    kept = sorted(inter)[:MAX_KEPT]
    return kept, note
KEPT, KEPT_NOTE = select_kept()
print(f"\nShared sentence set: {len(KEPT)} ({KEPT_NOTE})")
assert KEPT, "Shared set is empty — lower HIGH_SCORE_MIN or raise N_CANDIDATES."

# ---------------- PHASE 2: perturbed (all judges, same set) ------------------
rows = []
for cfg in JUDGES:
    try:
        model, tok = load_model(cfg)
        my_rows, jobs = [], []
        for row in perturb_table:
            if row["sent_id"] not in KEPT: continue
            en, zh = pairs[row["sent_id"]]
            r = dict(judge=cfg["name"], sent_id=row["sent_id"], source_en=en, original_zh=zh,
                     original_score=orig[cfg["name"]][row["sent_id"]],
                     original_raw=json.dumps(orig_raw[cfg["name"]][row["sent_id"]]["raw"], ensure_ascii=False),
                     fault=row["fault"], applied=row["perturbed"] is not None,
                     perturbed_zh=row["perturbed"] or "", modification=row["modification"],
                     fluency_gate=("n/a" if row["fault"] not in GATED_FAULTS else
                                   "passed" if row["fluent"] else "FAILED" if row["fluent"] is False else "unparsed"),
                     perturbed_score=None, perturbed_raw="", drop=None, adj_drop=None)
            my_rows.append(r)
            if r["applied"]: jobs.append((len(my_rows)-1, {"src": en, "cand": row["perturbed"]}))
        res = run_prompts(model, tok, cfg, jobs, JUDGE_PROMPT, parse_score, N_SAMPLES, TEMPERATURE,
                          f"[{cfg['name']}] perturbed", "judge")
        for idx, rec in res.items():
            my_rows[idx]["perturbed_score"] = median_score(rec)
            my_rows[idx]["perturbed_raw"] = json.dumps(rec["raw"], ensure_ascii=False)
            if my_rows[idx]["perturbed_score"] is not None:
                my_rows[idx]["drop"] = my_rows[idx]["original_score"] - my_rows[idx]["perturbed_score"]
        unload(model); rows += my_rows
    except Exception as e:
        print(f"[{cfg['name']}] FAILED in phase 2: {repr(e)[:300]}"); gc.collect(); torch.cuda.empty_cache()

results = pd.DataFrame(rows)
# fix 5: control-adjusted drop (per judge)
ctrl_mean = results[(results.fault == "control") & results.drop.notna()].groupby("judge")["drop"].mean().to_dict()
results["adj_drop"] = results.apply(lambda r: None if pd.isna(r.drop) else r.drop - ctrl_mean.get(r.judge, 0.0), axis=1)
results.to_csv("results_all_judges.csv", index=False)

# ----------------------------- STATS ---------------------------------
def boot_ci(vals, pred):                                        # fix 4: bootstrap CI on % meeting
    arr = np.asarray(vals, dtype=float)
    if len(arr) == 0: return (float("nan"), float("nan"))
    rng = np.random.default_rng(SEED)
    samp = rng.choice(arr, (BOOT_N, len(arr)), replace=True)
    pct = 100 * pred(samp).mean(axis=1)
    return float(np.percentile(pct, 2.5)), float(np.percentile(pct, 97.5))

summary, judge_health = [], {}
for cfg in JUDGES:
    R = results[results.judge == cfg["name"]]
    if R.empty: continue
    n_items = len(pairs) + int(R.applied.sum())
    n_unp = (len(pairs) - sum(v is not None for v in orig[cfg["name"]].values())) + int((R.applied & R.perturbed_score.isna()).sum())
    unp_pct = 100 * n_unp / n_items
    s = [v for v in orig[cfg["name"]].values() if v is not None]
    judge_health[cfg["name"]] = dict(unparsed_pct=round(unp_pct, 1),
                                     usable=unp_pct <= MAX_UNPARSED_PCT,
                                     orig_mean=round(statistics.mean(s), 2), orig_std=round(statistics.pstdev(s), 2),
                                     orig_min=min(s), orig_max=max(s), control_mean_drop=round(ctrl_mean.get(cfg["name"], 0), 2))
    for fault in FAULTS:
        d = R[R.fault == fault]
        app = d[d.applied & d.perturbed_score.notna()]
        n_gate_failed = 0
        if fault in GATED_FAULTS:
            n_gate_failed = int((app.fluency_gate == "FAILED").sum())
            app = app[app.fluency_gate != "FAILED"]
        vals = app["adj_drop"].tolist(); n_app = len(vals)
        if fault == "control":
            thr = CONTROL_MAX_DROP; raw_vals = d[d.applied & d.drop.notna()]["drop"].tolist()
            pred = lambda a, t=thr: a <= t; vals = raw_vals; n_app = len(vals)   # control judged on RAW drop
            expectation = f"raw drop <= {thr}"
        else:
            thr = EXPECTED_MIN_DROP[fault]; pred = lambda a, t=thr: a >= t
            expectation = f"control-adjusted drop >= {thr}"
        pct = 100 * np.mean(pred(np.asarray(vals, dtype=float))) if n_app else float("nan")
        lo, hi = boot_ci(vals, pred)
        if n_app < 5: verdict = "INSUFFICIENT DATA"
        elif fault == "control":
            verdict = "STABLE (good)" if pct >= RELIABLE_PCT and lo >= RELIABLE_CI_LO else \
                      "STABLE (low confidence)" if pct >= RELIABLE_PCT else "OVER-SENSITIVE"
        else:
            verdict = ("DETECTED RELIABLY" if pct >= RELIABLE_PCT and lo >= RELIABLE_CI_LO else
                       "LIKELY DETECTED (CI too wide)" if pct >= RELIABLE_PCT else
                       "PARTIAL" if pct >= 50 else "MISSED")
        summary.append(dict(judge=cfg["name"], fault=fault, n_applied=n_app,
                            n_skipped=int((~d.applied).sum()), n_unparsed=int((d.applied & d.perturbed_score.isna()).sum()),
                            n_gate_failed=n_gate_failed,
                            mean_raw_drop=round(app["drop"].mean(), 2) if len(app) else None,
                            mean_adj_drop=round(float(np.mean(vals)), 2) if n_app else None,
                            median_adj_drop=float(np.median(vals)) if n_app else None,
                            expectation=expectation, pct_meeting=round(pct, 1) if n_app else None,
                            ci95=f"[{lo:.0f}, {hi:.0f}]" if n_app else "", verdict=verdict))
summary = pd.DataFrame(summary)
summary.to_csv("summary_all_judges.csv", index=False)
comparison = summary.pivot(index="fault", columns="judge", values="pct_meeting").reindex(list(FAULTS))
comparison.to_csv("judge_comparison.csv")
for j in judge_health:
    print(f"\n== {j} ==  health: {judge_health[j]}")
    print(summary[summary.judge == j].drop(columns="judge").to_string(index=False))
print("\n% meeting expectation (shared set; control = % within 1 raw point)\n", comparison.to_string())

# ----------------------------- FINDING -------------------------------
def lists(S):
    f = lambda pref: S[(S.fault != "control") & S.verdict.str.startswith(pref)].fault.tolist()
    return f("DETECTED RELIABLY"), f("LIKELY"), f("PARTIAL"), f("MISSED")

def worst(j, fault, n=2):
    d = results[(results.judge == j) & (results.fault == fault) & results.adj_drop.notna() & (results.fluency_gate != "FAILED")]
    d = d.sort_values("adj_drop").head(n)
    return "\n".join(f"  - adj drop {r.adj_drop:.1f} (raw {int(r.drop)}): {r.modification}" for r in d.itertuples()) or "  (none)"

gate_total = sum(1 for r in perturb_table if r["sent_id"] in KEPT and r["fault"] in GATED_FAULTS and r["perturbed"])
gate_fail  = sum(1 for r in perturb_table if r["sent_id"] in KEPT and r["fault"] in GATED_FAULTS and r["fluent"] is False)

md = [f"# Sprint 54 · Task 1 — Judge fault-sensitivity finding (EN→CMN), v2\n",
      f"**Dataset:** {DATASET_NAME}  ",
      f"**Judges:** {', '.join(j for j in judge_health)} (free open weights, 4-bit, local via Unsloth)  ",
      f"**Scoring:** {N_SAMPLES} samples per item at T={TEMPERATURE}, median used; 0–10 scale.\n",
      f"**Method.** {len(pairs)} source/reference pairs were scored by every judge. The shared test set is "
      f"{len(KEPT)} sentences ({KEPT_NOTE}), so all judges are compared on identical material. Each sentence was "
      f"modified in exactly one documented way per fault and re-scored. Faults that cannot be applied are recorded as "
      f"skipped and excluded from the denominator (never counted as a pass). Drops for the five fault types are "
      f"**control-adjusted** (raw drop minus that judge's mean drop on the punctuation-only control) to remove "
      f"regression-to-the-mean from selecting high-scoring items. Negation/entity/clause edits were passed through a "
      f"meaning-blind grammar gate run by {JUDGES[FLUENCY_GATE_MODEL]['name'] if FLUENCY_GATE_MODEL < len(JUDGES) else 'the first judge'}: "
      f"{gate_fail}/{gate_total} edits failed and are excluded, so a detected fault reflects the meaning change rather than broken Chinese. "
      f"A fault counts as *reliably detected* when ≥ {RELIABLE_PCT}% of applied cases meet the expected minimum drop "
      f"**and** the 95% bootstrap CI lower bound is ≥ {RELIABLE_CI_LO}%.\n",
      "## Judge health\n", "| Judge | Unparsed % | Usable | Original-score mean / std / range | Mean control drop |", "|---|---|---|---|---|"]
md += [f"| {j} | {h['unparsed_pct']} | {'yes' if h['usable'] else f'**NO (> {MAX_UNPARSED_PCT}%)**'} | {h['orig_mean']} / {h['orig_std']} / {h['orig_min']}–{h['orig_max']} | {h['control_mean_drop']} |"
       for j, h in judge_health.items()]
md += ["\n## Cross-judge comparison — % of applied cases meeting expectation (control: % within 1 raw point)\n",
       comparison.to_markdown(), ""]
for j in judge_health:
    S = summary[summary.judge == j]; rel, likely, part, miss = lists(S); c = S[S.fault == "control"].iloc[0]
    md += [f"## Judge: {j}\n",
           "| Fault | Applied | Skipped | Unparsed | Gate-failed | Mean raw drop | Mean adj drop | Expectation | % meeting | 95% CI | Verdict |",
           "|---|---|---|---|---|---|---|---|---|---|---|"]
    md += [f"| {x.fault} | {x.n_applied} | {x.n_skipped} | {x.n_unparsed} | {x.n_gate_failed} | {x.mean_raw_drop} | {x.mean_adj_drop} | {x.expectation} | {x.pct_meeting} | {x.ci95} | **{x.verdict}** |"
           for x in S.itertuples()]
    md += ["", f"- **Reliably detected:** {', '.join(rel) or 'none'}",
           f"- **Likely detected, CI too wide (collect more data):** {', '.join(likely) or 'none'}",
           f"- **Partially detected:** {', '.join(part) or 'none'}",
           f"- **Missed:** {', '.join(miss) or 'none'}",
           f"- **Control:** {c.verdict} — {c.pct_meeting}% within {CONTROL_MAX_DROP} point, CI {c.ci95}", "",
           "Weakest cases (smallest adjusted drops):"]
    md += [f"**{f}**\n{worst(j, f)}" for f in FAULTS if f != "control"]
    md.append("")

ranking = sorted(((len(lists(summary[summary.judge == j])[0]), judge_health[j]["usable"],
                   summary[(summary.judge == j) & (summary.fault != "control")].pct_meeting.mean(), j) for j in judge_health), reverse=True)
best = ranking[0]
md += ["## Implication for the team",
       f"Best free judge in this run: **{best[3]}** ({best[0]}/5 faults reliably detected"
       f"{'' if best[1] else ', but flagged unusable on parse rate'}).",
       "Evaluation results from a judge can be trusted only for the faults listed as *reliably detected* for that judge. "
       "'Likely detected' means the point estimate is good but the sample is too small to be sure — rerun with a larger "
       "N_CANDIDATES before relying on it. For partial/missed faults, pair the judge with a deterministic check "
       "(negation-word alignment, number/NER consistency, length-ratio guard) or revise the judge prompt and re-run.", "",
       "Files: results_all_judges.csv (row-level incl. raw judge outputs), summary_all_judges.csv, judge_comparison.csv."]
open("FINDING.md", "w", encoding="utf-8").write("\n".join(md))
print("\n".join(md))

try:
    from google.colab import files
    for f in ["FINDING.md", "summary_all_judges.csv", "judge_comparison.csv", "results_all_judges.csv"]:
        files.download(f)
except Exception:
    pass

Dataset: WMT20 newstest en-zh (statmt.org via sacrebleu), 1418 segments
Candidates after filtering: 150
==((====))==  Unsloth 2026.10.7: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

[qwen3-8b] originals:   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=96) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

[qwen3-8b] fluency gate:   0%|          | 0/65 [00:00<?, ?it/s]

...

==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[qwen2.5-7b] perturbed:   0%|          | 0/56 [00:00<?, ?it/s]

Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

AttributeError: 'function' object has no attribute 'notna'

In [3]:
# =====================================================================
# Sprint 54 · Task 1 — Judge fault-sensitivity test (EN→CMN)  — v2.1
# Free open-weight judges via Unsloth. Colab T4 (free tier) is enough.
#
# v2.1: 'drop' column renamed 'raw_drop' (pandas name clash) · models load
# LAZILY — only if some prompt is not already in judge_cache_v2.json, so a
# re-run after a crash in the stats section loads/downloads nothing.
#
# Outputs: results_all_judges.csv, summary_all_judges.csv, judge_comparison.csv, FINDING.md
# =====================================================================
!pip -q install unsloth sacrebleu jieba pandas numpy tqdm tabulate

import os, re, json, random, hashlib, subprocess, statistics, gc, math
import numpy as np, torch, pandas as pd
from tqdm.auto import tqdm
import jieba, jieba.posseg as pseg
jieba.setLogLevel(60)

# ----------------------------- CONFIG --------------------------------
MODELS = [
    {"name": "qwen3-8b",   "id": "unsloth/Qwen3-8B-unsloth-bnb-4bit",
     "template_kwargs": {"enable_thinking": False}},
    {"name": "glm4-9b",    "id": "unsloth/GLM-4-9B-0414", "load_in_4bit": True},
    {"name": "qwen2.5-7b", "id": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"},
]
FLUENCY_GATE_MODEL = 0        # index into MODELS; runs the grammar gate once for everyone

N_CANDIDATES   = 150
HIGH_SCORE_MIN = 8
MAX_KEPT       = 60
MIN_KEPT       = 20
N_SAMPLES      = 3
TEMPERATURE    = 0.3
SEED           = 54
BATCH_SIZE     = 6
MAX_NEW_TOKENS = 96
MAX_SEQ_LEN    = 2048
CACHE_FILE     = "judge_cache_v2.json"
BOOT_N         = 2000

EXPECTED_MIN_DROP = {"untranslated": 7, "negation": 5, "entity": 4, "clause": 3, "truncation": 3}
CONTROL_MAX_DROP  = 1
RELIABLE_PCT      = 80
RELIABLE_CI_LO    = 65
MAX_UNPARSED_PCT  = 20
GATED_FAULTS      = {"negation", "entity", "clause"}

# >>> Paste your production judge prompt here. Keep {src} and {cand}. <<<
JUDGE_PROMPT = """You are an expert evaluator of English-to-Mandarin Chinese translation.

Rate the candidate translation on a 0-10 integer scale for how well it conveys the source
(accuracy of meaning, completeness, and fluency). 10 = a professional-quality translation;
0 = conveys none of the source meaning.

Source (English):
{src}

Candidate (Chinese):
{cand}

Respond with JSON only, no prose: {{"score": <integer 0-10>, "reason": "<one sentence>"}}"""

FLUENCY_PROMPT = """You will see one sentence in Chinese. Ignore its meaning and do not compare it to anything.
Judge only whether it is grammatical, natural Chinese that a native speaker could plausibly have written.

Sentence:
{cand}

Respond with JSON only: {{"fluent": true}} or {{"fluent": false}}"""

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --------------------------- DATASET ---------------------------------
def load_wmt(testset="wmt20", langpair="en-zh"):
    def echo(what):
        r = subprocess.run(["sacrebleu", "-t", testset, "-l", langpair, "--echo", what],
                           capture_output=True, text=True, check=True)
        return r.stdout.rstrip("\n").split("\n")
    src, ref = echo("src"), echo("ref")
    assert len(src) == len(ref) and len(src) > 100
    return list(zip(src, ref)), f"WMT20 newstest en-zh (statmt.org via sacrebleu), {len(src)} segments"

def load_opus100():
    from datasets import load_dataset
    ds = load_dataset("Helsinki-NLP/opus-100", "en-zh", split="test")
    return [(r["translation"]["en"], r["translation"]["zh"]) for r in ds], "OPUS-100 en-zh test (HuggingFace)"

try:
    all_pairs, DATASET_NAME = load_wmt()
except Exception as e:
    print("WMT20 load failed, falling back to OPUS-100:", repr(e)[:200])
    all_pairs, DATASET_NAME = load_opus100()

def usable(en, zh):
    return (8 <= len(en.split()) <= 45 and len(zh) >= 10
            and re.search(r"[\u4e00-\u9fff]", zh) is not None
            and not re.search(r"[A-Za-z]{3,}", zh))
pairs = [p for p in all_pairs if usable(*p)]
random.Random(SEED).shuffle(pairs)
pairs = pairs[:N_CANDIDATES]
print(f"Dataset: {DATASET_NAME}\nCandidates after filtering: {len(pairs)}")

# -------------------------- PERTURBATIONS ----------------------------
def stable_rng(*parts):
    h = hashlib.md5("||".join(map(str, parts)).encode()).hexdigest()
    return random.Random(int(h, 16) ^ SEED)

def tokens_with_pos(zh):
    toks, i = [], 0
    for t in pseg.lcut(zh):
        toks.append((t.word, t.flag, i)); i += len(t.word)
    return toks

NEG_TOKENS = {"不", "没", "没有", "无法", "并非", "未", "并不", "不会", "不能", "不是"}
def perturb_negation(zh, en):
    toks = tokens_with_pos(zh)
    for w, f, i in toks:
        if w in NEG_TOKENS:
            return zh[:i] + zh[i+len(w):], f"removed negation '{w}' at char {i}"
    for w, f, i in toks:
        if f.startswith("v") and f != "vn":
            neg = "没" if w.startswith("有") else "不"
            return zh[:i] + neg + zh[i:], f"inserted '{neg}' before verb '{w}' at char {i}"
    return None

CN_NUM = "二三四五六七八九"
CN_SWAP = {c: CN_NUM[(k + 3) % len(CN_NUM)] for k, c in enumerate(CN_NUM)}
ENTITY_POOL = {"nr": ["李明", "王芳", "张伟", "刘洋"],
               "ns": ["北京", "伦敦", "东京", "柏林", "巴黎", "开罗"],
               "nt": ["联合国", "欧盟", "世界银行", "微软", "国际奥委会"]}
def perturb_entity(zh, en):
    toks = tokens_with_pos(zh)
    for w, f, i in toks:
        if f == "m":
            m = re.search(r"\d+", w)
            if m:
                old = int(m.group())
                new = old + stable_rng(zh, w).choice([2, 3, 5]) * (1 if old < 50 else 10)
                new_w = w[:m.start()] + str(new) + w[m.end():]
                return zh[:i] + new_w + zh[i+len(w):], f"number '{w}' -> '{new_w}'"
            for ch in w:
                if ch in CN_SWAP:
                    new_w = w.replace(ch, CN_SWAP[ch], 1)
                    return zh[:i] + new_w + zh[i+len(w):], f"numeral '{w}' -> '{new_w}'"
    for w, f, i in toks:
        if f in ENTITY_POOL and len(w) >= 2:
            new_w = stable_rng(zh, w).choice([c for c in ENTITY_POOL[f] if c != w])
            return zh[:i] + new_w + zh[i+len(w):], f"entity[{f}] '{w}' -> '{new_w}'"
    return None

def perturb_clause(zh, en):
    clauses, buf = [], ""
    for ch in zh:
        buf += ch
        if ch in "，；：":
            clauses.append(buf); buf = ""
    if buf: clauses.append(buf)
    if len(clauses) < 2: return None
    droppable = [k for k, c in enumerate(clauses[:-1]) if len(c.strip("，；：")) >= 4]
    if not droppable: return None
    k = droppable[len(droppable) // 2]
    return "".join(c for j, c in enumerate(clauses) if j != k), f"dropped clause #{k+1}/{len(clauses)}: '{clauses[k]}'"

def perturb_truncation(zh, en):
    if len(zh) < 14: return None
    cut = int(len(zh) * 0.6)
    return zh[:cut], f"kept first {cut}/{len(zh)} chars (dropped {len(zh)-cut})"

def perturb_untranslated(zh, en):
    return en, "returned the English source verbatim as the output"

def perturb_control(zh, en):
    if "，" in zh:        return zh.replace("，", ",", 1), "first full-width comma '，' -> ASCII ','"
    if zh.endswith("。"): return zh[:-1] + ".", "final '。' -> ASCII '.'"
    return zh + "。", "appended final '。'"

FAULTS = {"untranslated": perturb_untranslated, "negation": perturb_negation, "entity": perturb_entity,
          "clause": perturb_clause, "truncation": perturb_truncation, "control": perturb_control}

perturb_table = []
for k, (en, zh) in enumerate(pairs):
    for fault, fn in FAULTS.items():
        res = fn(zh, en)
        perturb_table.append(dict(sent_id=k, fault=fault, perturbed=res[0] if res else None,
                                  modification=res[1] if res else "SKIPPED: fault not applicable to this sentence",
                                  fluent=None))

# ----------------------------- MODELS --------------------------------
_cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
print(f"Cache entries on disk: {len(_cache)}")

def load_model(cfg):
    assert torch.cuda.is_available(), "Uncached prompts remain but no GPU — switch the runtime to a GPU."
    try:
        from unsloth import FastLanguageModel, FastModel
        loader = FastModel if cfg.get("loader") == "FastModel" else FastLanguageModel
        model, tok = loader.from_pretrained(model_name=cfg["id"], max_seq_length=MAX_SEQ_LEN,
                                            load_in_4bit=cfg.get("load_in_4bit", True), dtype=None)
        loader.for_inference(model)
    except Exception as e:
        print(f"[{cfg['name']}] Unsloth loader failed ({repr(e)[:150]}) → transformers 4-bit fallback")
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        tok = AutoTokenizer.from_pretrained(cfg["id"])
        model = AutoModelForCausalLM.from_pretrained(cfg["id"], quantization_config=bnb, device_map="auto")
        model.eval()
    if getattr(tok, "pad_token", None) is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return model, tok

class LazyModel:
    """Loads the model only on first real use; a fully-cached pass never loads it."""
    def __init__(self, cfg): self.cfg, self.model, self.tok = cfg, None, None
    def get(self):
        if self.model is None:
            print(f"[{self.cfg['name']}] loading model (uncached prompts remain)")
            self.model, self.tok = load_model(self.cfg)
        return self.model, self.tok
    def close(self):
        if self.model is not None:
            del self.model; self.model = self.tok = None
            gc.collect(); torch.cuda.empty_cache()

def generate(model, tok, prompts, cfg, n=1, temperature=0.0):
    texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                     add_generation_prompt=True, **cfg.get("template_kwargs", {})) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id, num_return_sequences=n)
    kw.update(dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    dec = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [dec[i*n:(i+1)*n] for i in range(len(prompts))]

def parse_score(txt):
    txt = re.sub(r"<think>.*?</think>", "", txt, flags=re.S)
    m = re.search(r"\{.*?\}", txt, re.S)
    try:
        return max(0, min(10, int(round(float(json.loads(m.group(0))["score"])))))
    except Exception:
        m = re.search(r'"?score"?\s*[:=]\s*(\d+)', txt)
        return max(0, min(10, int(m.group(1)))) if m else None

def parse_fluent(txt):
    m = re.search(r'"?fluent"?\s*:\s*(true|false)', txt, re.I)
    return None if not m else m.group(1).lower() == "true"

def run_prompts(lm, jobs, prompt_tmpl, parser, n, temperature, desc, tag):
    cfg = lm.cfg; out, todo = {}, []
    for jid, fields in jobs:
        key = hashlib.sha256(f"{tag}|{cfg['id']}|{prompt_tmpl}|{n}|{temperature}|{json.dumps(fields, ensure_ascii=False)}".encode()).hexdigest()
        if key in _cache: out[jid] = _cache[key]
        else: todo.append((jid, key, prompt_tmpl.format(**fields)))
    if not todo:
        print(f"{desc}: all {len(jobs)} prompts served from cache"); return out
    model, tok = lm.get()
    todo.sort(key=lambda t: len(t[2]))
    for b in tqdm(range(0, len(todo), BATCH_SIZE), desc=desc, leave=False):
        batch = todo[b:b+BATCH_SIZE]
        for (jid, key, _), raws in zip(batch, generate(model, tok, [t[2] for t in batch], cfg, n, temperature)):
            rec = {"vals": [parser(r) for r in raws], "raw": raws}
            _cache[key] = rec; out[jid] = rec
    json.dump(_cache, open(CACHE_FILE, "w"), ensure_ascii=False)
    return out

def median_score(rec):
    vals = [v for v in rec["vals"] if v is not None]
    return statistics.median(vals) if len(vals) >= math.ceil(len(rec["vals"]) / 2) else None

# ---------------- PHASE 1: originals (all judges) + fluency gate -------------
orig, orig_raw = {}, {}
for mi, cfg in enumerate(MODELS):
    lm = LazyModel(cfg)
    try:
        res = run_prompts(lm, [(k, {"src": en, "cand": zh}) for k, (en, zh) in enumerate(pairs)],
                          JUDGE_PROMPT, parse_score, N_SAMPLES, TEMPERATURE, f"[{cfg['name']}] originals", "judge")
        orig[cfg["name"]] = {k: median_score(r) for k, r in res.items()}
        orig_raw[cfg["name"]] = res
        if mi == FLUENCY_GATE_MODEL:
            gate_jobs = [(i, {"cand": row["perturbed"]}) for i, row in enumerate(perturb_table)
                         if row["perturbed"] is not None and row["fault"] in GATED_FAULTS]
            gres = run_prompts(lm, gate_jobs, FLUENCY_PROMPT, parse_fluent, 1, 0.0,
                               f"[{cfg['name']}] fluency gate", "gate")
            for i, r in gres.items(): perturb_table[i]["fluent"] = r["vals"][0]
        s = [v for v in orig[cfg["name"]].values() if v is not None]
        print(f"[{cfg['name']}] originals: mean {statistics.mean(s):.2f}, std {statistics.pstdev(s):.2f}, "
              f"min {min(s)}, max {max(s)}, >= {HIGH_SCORE_MIN}: {sum(v >= HIGH_SCORE_MIN for v in s)}, unparsed {len(pairs)-len(s)}")
    except Exception as e:
        print(f"[{cfg['name']}] FAILED in phase 1: {repr(e)[:300]}")
    finally:
        lm.close()
assert orig, "No judge completed phase 1."
JUDGES = [c for c in MODELS if c["name"] in orig]

def select_kept():
    high = {c["name"]: {k for k, v in orig[c["name"]].items() if v is not None and v >= HIGH_SCORE_MIN} for c in JUDGES}
    inter = set.intersection(*high.values())
    note = f"sentences scored >= {HIGH_SCORE_MIN} by EVERY judge"
    if len(inter) < MIN_KEPT:
        inter = {k for k in range(len(pairs))
                 if all(orig[c["name"]].get(k) is not None for c in JUDGES)
                 and statistics.mean(orig[c["name"]][k] for c in JUDGES) >= HIGH_SCORE_MIN}
        note = f"intersection too small; widened to sentences with MEAN judge score >= {HIGH_SCORE_MIN}"
    return sorted(inter)[:MAX_KEPT], note
KEPT, KEPT_NOTE = select_kept()
print(f"\nShared sentence set: {len(KEPT)} ({KEPT_NOTE})")
assert KEPT, "Shared set is empty — lower HIGH_SCORE_MIN or raise N_CANDIDATES."

# ---------------- PHASE 2: perturbed (all judges, same set) ------------------
rows = []
for cfg in JUDGES:
    lm = LazyModel(cfg)
    try:
        my_rows, jobs = [], []
        for row in perturb_table:
            if row["sent_id"] not in KEPT: continue
            en, zh = pairs[row["sent_id"]]
            r = dict(judge=cfg["name"], sent_id=row["sent_id"], source_en=en, original_zh=zh,
                     original_score=orig[cfg["name"]][row["sent_id"]],
                     original_raw=json.dumps(orig_raw[cfg["name"]][row["sent_id"]]["raw"], ensure_ascii=False),
                     fault=row["fault"], applied=row["perturbed"] is not None,
                     perturbed_zh=row["perturbed"] or "", modification=row["modification"],
                     fluency_gate=("n/a" if row["fault"] not in GATED_FAULTS else
                                   "passed" if row["fluent"] else "FAILED" if row["fluent"] is False else "unparsed"),
                     perturbed_score=None, perturbed_raw="", raw_drop=None, adj_drop=None)
            my_rows.append(r)
            if r["applied"]: jobs.append((len(my_rows)-1, {"src": en, "cand": row["perturbed"]}))
        res = run_prompts(lm, jobs, JUDGE_PROMPT, parse_score, N_SAMPLES, TEMPERATURE,
                          f"[{cfg['name']}] perturbed", "judge")
        for idx, rec in res.items():
            my_rows[idx]["perturbed_score"] = median_score(rec)
            my_rows[idx]["perturbed_raw"] = json.dumps(rec["raw"], ensure_ascii=False)
            if my_rows[idx]["perturbed_score"] is not None:
                my_rows[idx]["raw_drop"] = my_rows[idx]["original_score"] - my_rows[idx]["perturbed_score"]
        rows += my_rows
    except Exception as e:
        print(f"[{cfg['name']}] FAILED in phase 2: {repr(e)[:300]}")
    finally:
        lm.close()

results = pd.DataFrame(rows)
ctrl_mean = results[(results.fault == "control") & results["raw_drop"].notna()].groupby("judge")["raw_drop"].mean().to_dict()
results["adj_drop"] = [None if pd.isna(d) else d - ctrl_mean.get(j, 0.0) for d, j in zip(results["raw_drop"], results["judge"])]
results.to_csv("results_all_judges.csv", index=False)

# ----------------------------- STATS ---------------------------------
def boot_ci(vals, pred):
    arr = np.asarray(vals, dtype=float)
    if len(arr) == 0: return (float("nan"), float("nan"))
    rng = np.random.default_rng(SEED)
    samp = rng.choice(arr, (BOOT_N, len(arr)), replace=True)
    pct = 100 * pred(samp).mean(axis=1)
    return float(np.percentile(pct, 2.5)), float(np.percentile(pct, 97.5))

summary, judge_health = [], {}
for cfg in JUDGES:
    R = results[results.judge == cfg["name"]]
    if R.empty: continue
    n_items = len(pairs) + int(R.applied.sum())
    n_unp = (len(pairs) - sum(v is not None for v in orig[cfg["name"]].values())) + int((R.applied & R.perturbed_score.isna()).sum())
    unp_pct = 100 * n_unp / n_items
    s = [v for v in orig[cfg["name"]].values() if v is not None]
    judge_health[cfg["name"]] = dict(unparsed_pct=round(unp_pct, 1), usable=unp_pct <= MAX_UNPARSED_PCT,
                                     orig_mean=round(statistics.mean(s), 2), orig_std=round(statistics.pstdev(s), 2),
                                     orig_min=min(s), orig_max=max(s), control_mean_drop=round(ctrl_mean.get(cfg["name"], 0), 2))
    for fault in FAULTS:
        d = R[R.fault == fault]
        app = d[d.applied & d.perturbed_score.notna()]
        n_gate_failed = 0
        if fault in GATED_FAULTS:
            n_gate_failed = int((app.fluency_gate == "FAILED").sum())
            app = app[app.fluency_gate != "FAILED"]
        if fault == "control":
            thr = CONTROL_MAX_DROP; vals = app["raw_drop"].tolist()
            pred = lambda a, t=thr: a <= t; expectation = f"raw drop <= {thr}"
        else:
            thr = EXPECTED_MIN_DROP[fault]; vals = app["adj_drop"].tolist()
            pred = lambda a, t=thr: a >= t; expectation = f"control-adjusted drop >= {thr}"
        n_app = len(vals)
        pct = 100 * np.mean(pred(np.asarray(vals, dtype=float))) if n_app else float("nan")
        lo, hi = boot_ci(vals, pred)
        if n_app < 5: verdict = "INSUFFICIENT DATA"
        elif fault == "control":
            verdict = ("STABLE (good)" if pct >= RELIABLE_PCT and lo >= RELIABLE_CI_LO else
                       "STABLE (low confidence)" if pct >= RELIABLE_PCT else "OVER-SENSITIVE")
        else:
            verdict = ("DETECTED RELIABLY" if pct >= RELIABLE_PCT and lo >= RELIABLE_CI_LO else
                       "LIKELY DETECTED (CI too wide)" if pct >= RELIABLE_PCT else
                       "PARTIAL" if pct >= 50 else "MISSED")
        summary.append(dict(judge=cfg["name"], fault=fault, n_applied=n_app,
                            n_skipped=int((~d.applied).sum()), n_unparsed=int((d.applied & d.perturbed_score.isna()).sum()),
                            n_gate_failed=n_gate_failed,
                            mean_raw_drop=round(app["raw_drop"].mean(), 2) if len(app) else None,
                            mean_adj_drop=round(float(np.mean(vals)), 2) if n_app else None,
                            median_adj_drop=float(np.median(vals)) if n_app else None,
                            expectation=expectation, pct_meeting=round(pct, 1) if n_app else None,
                            ci95=f"[{lo:.0f}, {hi:.0f}]" if n_app else "", verdict=verdict))
summary = pd.DataFrame(summary)
summary.to_csv("summary_all_judges.csv", index=False)
comparison = summary.pivot(index="fault", columns="judge", values="pct_meeting").reindex(list(FAULTS))
comparison.to_csv("judge_comparison.csv")
for j in judge_health:
    print(f"\n== {j} ==  health: {judge_health[j]}")
    print(summary[summary.judge == j].drop(columns="judge").to_string(index=False))
print("\n% meeting expectation (shared set; control = % within 1 raw point)\n", comparison.to_string())

# ----------------------------- FINDING -------------------------------
def lists(S):
    f = lambda pref: S[(S.fault != "control") & S.verdict.str.startswith(pref)].fault.tolist()
    return f("DETECTED RELIABLY"), f("LIKELY"), f("PARTIAL"), f("MISSED")

def worst(j, fault, n=2):
    d = results[(results.judge == j) & (results.fault == fault) & results.adj_drop.notna() & (results.fluency_gate != "FAILED")]
    d = d.sort_values("adj_drop").head(n)
    return "\n".join(f"  - adj drop {r.adj_drop:.1f} (raw {int(r.raw_drop)}): {r.modification}" for r in d.itertuples()) or "  (none)"

gate_total = sum(1 for r in perturb_table if r["sent_id"] in KEPT and r["fault"] in GATED_FAULTS and r["perturbed"])
gate_fail  = sum(1 for r in perturb_table if r["sent_id"] in KEPT and r["fault"] in GATED_FAULTS and r["fluent"] is False)
gate_name  = MODELS[FLUENCY_GATE_MODEL]["name"]

md = [f"# Sprint 54 · Task 1 — Judge fault-sensitivity finding (EN→CMN), v2\n",
      f"**Dataset:** {DATASET_NAME}  ",
      f"**Judges:** {', '.join(j for j in judge_health)} (free open weights, 4-bit, local via Unsloth)  ",
      f"**Scoring:** {N_SAMPLES} samples per item at T={TEMPERATURE}, median used; 0–10 scale.\n",
      f"**Method.** {len(pairs)} source/reference pairs were scored by every judge. The shared test set is "
      f"{len(KEPT)} sentences ({KEPT_NOTE}), so all judges are compared on identical material. Each sentence was "
      f"modified in exactly one documented way per fault and re-scored. Faults that cannot be applied are recorded as "
      f"skipped and excluded from the denominator (never counted as a pass). Drops for the five fault types are "
      f"**control-adjusted** (raw drop minus that judge's mean drop on the punctuation-only control) to remove "
      f"regression-to-the-mean from selecting high-scoring items. Negation/entity/clause edits were passed through a "
      f"meaning-blind grammar gate run by {gate_name}: {gate_fail}/{gate_total} edits failed and are excluded, so a "
      f"detected fault reflects the meaning change rather than broken Chinese. A fault counts as *reliably detected* "
      f"when ≥ {RELIABLE_PCT}% of applied cases meet the expected minimum drop **and** the 95% bootstrap CI lower bound "
      f"is ≥ {RELIABLE_CI_LO}%.\n",
      "## Judge health\n", "| Judge | Unparsed % | Usable | Original-score mean / std / range | Mean control drop |", "|---|---|---|---|---|"]
md += [f"| {j} | {h['unparsed_pct']} | {'yes' if h['usable'] else f'**NO (> {MAX_UNPARSED_PCT}%)**'} | {h['orig_mean']} / {h['orig_std']} / {h['orig_min']}–{h['orig_max']} | {h['control_mean_drop']} |"
       for j, h in judge_health.items()]
md += ["\n## Cross-judge comparison — % of applied cases meeting expectation (control: % within 1 raw point)\n",
       comparison.to_markdown(), ""]
for j in judge_health:
    S = summary[summary.judge == j]; rel, likely, part, miss = lists(S); c = S[S.fault == "control"].iloc[0]
    md += [f"## Judge: {j}\n",
           "| Fault | Applied | Skipped | Unparsed | Gate-failed | Mean raw drop | Mean adj drop | Expectation | % meeting | 95% CI | Verdict |",
           "|---|---|---|---|---|---|---|---|---|---|---|"]
    md += [f"| {x.fault} | {x.n_applied} | {x.n_skipped} | {x.n_unparsed} | {x.n_gate_failed} | {x.mean_raw_drop} | {x.mean_adj_drop} | {x.expectation} | {x.pct_meeting} | {x.ci95} | **{x.verdict}** |"
           for x in S.itertuples()]
    md += ["", f"- **Reliably detected:** {', '.join(rel) or 'none'}",
           f"- **Likely detected, CI too wide (collect more data):** {', '.join(likely) or 'none'}",
           f"- **Partially detected:** {', '.join(part) or 'none'}",
           f"- **Missed:** {', '.join(miss) or 'none'}",
           f"- **Control:** {c.verdict} — {c.pct_meeting}% within {CONTROL_MAX_DROP} point, CI {c.ci95}", "",
           "Weakest cases (smallest adjusted drops):"]
    md += [f"**{f}**\n{worst(j, f)}" for f in FAULTS if f != "control"]
    md.append("")

ranking = sorted(((len(lists(summary[summary.judge == j])[0]), judge_health[j]["usable"],
                   summary[(summary.judge == j) & (summary.fault != "control")].pct_meeting.mean(), j) for j in judge_health), reverse=True)
best = ranking[0]
md += ["## Implication for the team",
       f"Best free judge in this run: **{best[3]}** ({best[0]}/5 faults reliably detected"
       f"{'' if best[1] else ', but flagged unusable on parse rate'}).",
       "Evaluation results from a judge can be trusted only for the faults listed as *reliably detected* for that judge. "
       "'Likely detected' means the point estimate is good but the sample is too small to be sure — rerun with a larger "
       "N_CANDIDATES before relying on it. For partial/missed faults, pair the judge with a deterministic check "
       "(negation-word alignment, number/NER consistency, length-ratio guard) or revise the judge prompt and re-run.", "",
       "Files: results_all_judges.csv (row-level incl. raw judge outputs), summary_all_judges.csv, judge_comparison.csv."]
open("FINDING.md", "w", encoding="utf-8").write("\n".join(md))
print("\n".join(md))

try:
    from google.colab import files
    for f in ["FINDING.md", "summary_all_judges.csv", "judge_comparison.csv", "results_all_judges.csv"]:
        files.download(f)
except Exception:
    pass

Dataset: WMT20 newstest en-zh (statmt.org via sacrebleu), 1418 segments
Candidates after filtering: 150
Cache entries on disk: 1831
[qwen3-8b] originals: all 150 prompts served from cache
[qwen3-8b] fluency gate: all 385 prompts served from cache
[qwen3-8b] originals: mean 8.24, std 1.23, min 5, max 10, >= 8: 116, unparsed 0
[glm4-9b] originals: all 150 prompts served from cache
[glm4-9b] originals: mean 9.20, std 0.65, min 7, max 10, >= 8: 149, unparsed 0
[qwen2.5-7b] originals: all 150 prompts served from cache
[qwen2.5-7b] originals: mean 8.10, std 0.90, min 2, max 10, >= 8: 129, unparsed 0

Shared sentence set: 60 (sentences scored >= 8 by EVERY judge)
[qwen3-8b] perturbed: all 332 prompts served from cache
[glm4-9b] perturbed: all 332 prompts served from cache
[qwen2.5-7b] perturbed: all 332 prompts served from cache

== qwen3-8b ==  health: {'unparsed_pct': 0.0, 'usable': True, 'orig_mean': 8.24, 'orig_std': 1.23, 'orig_min': 5, 'orig_max': 10, 'control_mean_drop': 0.07}
       f

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>